In [ ]:
import sys
sys.path.insert(0, "../") # replace with path/to/project/root
from data import get_data, get_site_ids

In [ ]:
# my training site
uid = "USGS-06604440"
data = get_data(site_uid=uid) 

In [ ]:
print(data.surplus.columns)
print(data.crops.columns)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import numpy as np

uid = "USGS-05482500"  # replace with your site uid
data = get_data(uid)

# --- aggregate water and rain to weekly ---
agg_data = data.aggregate_by_interval(interval="1W")

nitrate_weekly = data.water["nitrate_con"].resample("1W").mean()
rain_weekly = agg_data.rain.groupby("date")["precip_1w"].mean().reset_index()
rain_weekly.columns = ["date", "precip_avg"]

# --- get year from nitrate index to join annual data ---
nitrate_df = nitrate_weekly.reset_index()
nitrate_df.columns = ["date", "nitrate_avg"]
nitrate_df["year"] = nitrate_df["date"].dt.year

# --- aggregate surplus annually ---
surplus_annual = data.surplus.groupby("year")["surplus_kgha"].mean().reset_index()

# --- aggregate crops annually ---
crops_annual = data.crops.groupby("year").mean().reset_index()


# --- merge everything on year ---
nitrate_df["date"] = nitrate_df["date"].dt.tz_localize(None).astype("datetime64[ms]")
rain_weekly["date"] = rain_weekly["date"].astype("datetime64[ms]")

df = nitrate_df \
    .merge(surplus_annual, on="year", how="left") \
    .merge(crops_annual, on="year", how="left") \
    .merge(rain_weekly, on="date", how="left")

df = df.dropna()

# --- define features and target ---
feature_cols = [c for c in df.columns if c not in ["date", "nitrate_avg", "node_id"]]
X = df[feature_cols]
y = df["nitrate_avg"]

# --- train/test split and fit ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

# --- evaluate ---
y_pred = model.predict(X_test)
print(f"R²:   {r2_score(y_test, y_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")

# --- coefficients ---
coef_df = pd.DataFrame({"feature": feature_cols, "coefficient": model.coef_})
print(coef_df.sort_values("coefficient", ascending=False))

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score
import pandas as pd

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Dummy (mean)":        DummyRegressor(strategy="mean"),
    "Linear Regression":   LinearRegression(),
    "Ridge":               Ridge(),
    "Lasso":               Lasso(),
    "Random Forest":       RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting":   GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = []
for name, m in models.items():
    cv_scores = cross_val_score(m, X_train_scaled, y_train, cv=5, scoring="r2")
    m.fit(X_train_scaled, y_train)
    test_r2 = r2_score(y_test, m.predict(X_test_scaled))
    test_rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_test_scaled)))
    results.append({
        "model":       name,
        "cv_r2_mean":  cv_scores.mean(),
        "cv_r2_std":   cv_scores.std(),
        "test_r2":     test_r2,
        "test_rmse":   test_rmse,
    })

results_df = pd.DataFrame(results).sort_values("test_r2", ascending=False)
print(results_df.to_string(index=False))